# 02 ? Farm A preprocessing

This notebook creates a separate, compressed processed copy of Wind Farm A. It never changes files under `data/raw/`.

**Simple rules:** keep zero values unchanged; leave missing sensor readings blank (do not fill them); use only documented average sensors; keep each event and its train/prediction split; start a new time segment at gaps longer than 10 minutes or any split boundary; flag timestamps shared between prediction and another event?s training data. Later, an LSTM window containing any missing sensor value should be skipped.

This is a first model-preparation version, not a claim that every sensor?s physical meaning is known.

## Why these choices?

- **Missing readings:** keep blank. Filling them would invent measurements. Later window creation can skip any window that contains a blank. Risk: some windows near missing readings will not be usable.
- **Zeros:** keep them as recorded. The dataset does not say that zero means missing. Risk: a particular sensor could use zero as a special code, but we have no evidence of that yet.
- **Long gaps:** a new `segment_id` begins after more than 10 minutes. Ten minutes is the expected cadence and a practical initial cutoff. Risk: this may split around harmless delays; we can review segment lengths later.
- **Train/prediction boundary:** always starts a new segment. A future window must stay inside one split, event, and segment.
- **Constant sensors:** keep them and report them based on training rows. Do not drop based on zeros alone. Feature selection is a later decision.
- **Overlapping timestamps:** flag them in the output. Farm A has 38,367 prediction rows sharing a turbine/timestamp with another event?s training rows. This needs a specific evaluation rule before model metrics are trusted.
- **Labels and units:** preserve both CSV `status_type_id` and metadata `event_label`. The README and metadata disagree on overall v6 totals, and some units are unclear; this notebook does not guess which source is right.
- **Scaling:** not done here. A scaler must learn its settings from training data only.

In [5]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

cwd=Path.cwd()
PROJECT_ROOT=next((p for p in (cwd,*cwd.parents) if (p/"data/raw/Wind Farm A/datasets").is_dir()),None)
if PROJECT_ROOT is None: raise FileNotFoundError("Cannot find data/raw/Wind Farm A/datasets")
FARM_DIR=PROJECT_ROOT/"data/raw/Wind Farm A"; DATA_DIR=FARM_DIR/"datasets"
OUTPUT_DIR=PROJECT_ROOT/"data/processed/farm_a"
EVENT_OUTPUT_DIR=OUTPUT_DIR/"events"
EVENTS=pd.read_csv(FARM_DIR/"event_info.csv",sep=";")
FEATURES=pd.read_csv(FARM_DIR/"feature_description.csv",sep=";")
FILES=sorted(DATA_DIR.glob("*.csv"),key=lambda p:int(p.stem) if p.stem.isdigit() else p.stem)
print("Farm A source files:",len(FILES))
print("Processed outputs will go to:",OUTPUT_DIR)
print("Output format: one compressed CSV per event, plus two small summary CSV files.")

Farm A source files: 22
Processed outputs will go to: c:\Users\mohamad\Desktop\Predictive-Maintenance\data\processed\farm_a
Output format: one compressed CSV per event, plus two small summary CSV files.


## 1. Choose the sensor columns and verify inputs

The feature-description table labels the average readings. Some CSV headers end in `_avg`; some average channels use a bare sensor name. We match either form and show anything unmatched. We also verify that every event file has one metadata row, the expected columns, valid timestamps, and only the supplied `train` / `prediction` split labels.

In [6]:
STRUCTURAL=["time_stamp","asset_id","id","train_test","status_type_id"]
FILES_HEADER=pd.read_csv(FILES[0],sep=";",nrows=0).columns.tolist()
average_rows=FEATURES.loc[FEATURES.statistics_type.astype(str).str.strip().str.lower().eq("average")]
feature_map=[]
for r in average_rows.itertuples(index=False):
 base=str(r.sensor_name); col=next((c for c in (base+"_avg",base) if c in FILES_HEADER),None)
 feature_map.append({"sensor_name":base,"csv_column":col,"description":r.description,"unit":r.unit})
FEATURE_MAP=pd.DataFrame(feature_map)
SENSORS=FEATURE_MAP.csv_column.dropna().drop_duplicates().tolist()
print("Average descriptions:",len(average_rows),"| matched CSV columns:",len(SENSORS))
print("Unmatched descriptions:"); display(FEATURE_MAP.loc[FEATURE_MAP.csv_column.isna()])
meta=EVENTS.copy(); meta["event_key"]=meta.event_id.astype(str); META=meta.set_index("event_key").to_dict("index")
file_ids={f.stem for f in FILES}; metadata_ids=set(meta.event_key)
if file_ids != metadata_ids: raise ValueError(f"File/metadata ID mismatch. Missing metadata: {sorted(file_ids-metadata_ids)}; missing files: {sorted(metadata_ids-file_ids)}")
for f in FILES:
 header=pd.read_csv(f,sep=";",nrows=0).columns.tolist()
 if header != FILES_HEADER: raise ValueError(f"Unexpected column schema in {f.name}")
 if not set(STRUCTURAL+SENSORS).issubset(header): raise ValueError(f"Required columns missing in {f.name}")
 d=pd.read_csv(f,sep=";",usecols=STRUCTURAL)
 if not d.asset_id.astype(str).eq(str(META[f.stem]["asset"])).all(): raise ValueError(f"Turbine metadata mismatch in {f.name}")
 if not set(d.train_test.dropna().unique()).issubset({"train","prediction"}): raise ValueError(f"Unexpected train_test value in {f.name}")
 if d.time_stamp.isna().any(): raise ValueError(f"Missing timestamp in {f.name}; stop and review before export")
 if d.train_test.isna().any(): raise ValueError(f"Missing train_test split in {f.name}; stop and review before export")
print("Preflight passed for all",len(FILES),"event files.")
print("Selected sensor columns:",SENSORS)

Average descriptions: 45 | matched CSV columns: 45
Unmatched descriptions:


,sensor_name,csv_column,description,unit


Preflight passed for all 22 event files.
Selected sensor columns: ['sensor_0_avg', 'sensor_1_avg', 'sensor_2_avg', 'wind_speed_4_avg', 'sensor_6_avg', 'sensor_7_avg', 'sensor_8_avg', 'sensor_9_avg', 'sensor_10_avg', 'sensor_11_avg', 'sensor_12_avg', 'sensor_13_avg', 'sensor_14_avg', 'sensor_15_avg', 'sensor_16_avg', 'sensor_17_avg', 'sensor_19_avg', 'sensor_20_avg', 'sensor_21_avg', 'sensor_22_avg', 'sensor_23_avg', 'sensor_24_avg', 'sensor_25_avg', 'sensor_26_avg', 'sensor_32_avg', 'sensor_33_avg', 'sensor_34_avg', 'sensor_35_avg', 'sensor_36_avg', 'sensor_37_avg', 'sensor_38_avg', 'sensor_39_avg', 'sensor_40_avg', 'sensor_41_avg', 'sensor_42_avg', 'sensor_43_avg', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'sensor_53_avg']


## 2. Find timestamps shared across event splits

We compare `(turbine, timestamp)` across files. If the same time is a prediction row in one event and a training row in another, we mark that key in the processed files. We keep the rows for now so the original supplied data is represented; the model evaluation step must decide how to exclude or group them.

In [7]:
train_event_ids=defaultdict(set); prediction_event_ids=defaultdict(set)
for f in FILES:
 d=pd.read_csv(f,sep=";",usecols=["asset_id","time_stamp","train_test"])
 d["time_stamp"]=pd.to_datetime(d.time_stamp,errors="coerce")
 for a,t in zip(d.loc[d.train_test.eq("train"),"asset_id"].astype(str),d.loc[d.train_test.eq("train"),"time_stamp"]): train_event_ids[(a,t)].add(f.stem)
 for a,t in zip(d.loc[d.train_test.eq("prediction"),"asset_id"].astype(str),d.loc[d.train_test.eq("prediction"),"time_stamp"]): prediction_event_ids[(a,t)].add(f.stem)
OVERLAP_KEYS={key for key,pred_ids in prediction_event_ids.items() if train_event_ids.get(key,set())-pred_ids}
OVERLAP_PREDICTION_ROWS=sum(len(prediction_event_ids[key]) for key in OVERLAP_KEYS)
print("Shared turbine/timestamp keys:",len(OVERLAP_KEYS))
print("Prediction event rows affected:",OVERLAP_PREDICTION_ROWS)
print("Example keys:")
display(pd.DataFrame([(a,t,sorted(prediction_event_ids[(a,t)]),sorted(train_event_ids[(a,t)])) for a,t in list(OVERLAP_KEYS)[:10]],columns=["turbine","timestamp","prediction_events","training_events"]))

Shared turbine/timestamp keys: 38367
Prediction event rows affected: 38367
Example keys:


,turbine,timestamp,prediction_events,training_events
0,10,2023-05-06 23:10:00,[3],"[10, 17, 42]"
1,10,2023-01-22 20:50:00,[40],"[10, 17, 3, 42]"
2,21,2023-05-20 10:10:00,[13],"[22, 51, 72]"
3,11,2023-08-02 03:50:00,[68],[69]
4,0,2023-08-18 15:00:00,[0],[26]
5,13,2023-03-04 21:30:00,[14],"[38, 45, 84]"
6,11,2023-06-01 08:50:00,[25],"[68, 69]"
7,10,2023-05-04 20:20:00,[3],"[10, 17, 42]"
8,11,2023-08-06 08:10:00,[68],[69]
9,10,2023-01-27 01:20:00,[40],"[10, 17, 3, 42]"


## 3. Prepare and save Farm A event files

The output keeps the event identity, metadata label, status code, supplied split, sensor values, source row order, segment number, and overlap flag. Missing sensor values remain blank in the compressed CSV. Zeros remain zero. We do not make LSTM windows or scale the values here.

In [8]:
EVENT_OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
event_reports=[]; sensor_counts={c:{"missing":0,"zeros":0,"non_numeric":0,"train_count":0,"train_min":np.inf,"train_max":-np.inf} for c in SENSORS}
for f in FILES:
 eid=f.stem; m=META[eid]; raw=pd.read_csv(f,sep=";")
 d=raw[STRUCTURAL+SENSORS].copy(); d["source_row_order"]=np.arange(len(d))
 d["time_stamp"]=pd.to_datetime(d.time_stamp,errors="coerce")
 non_numeric_total=0
 for col in SENSORS:
  before=d[col]; numeric=pd.to_numeric(before,errors="coerce")
  bad=int((numeric.isna()&before.notna()).sum()); non_numeric_total+=bad
  sensor_counts[col]["non_numeric"]+=bad; d[col]=numeric
 d["farm"]="A"; d["event_id"]=eid; d["event_label"]=str(m["event_label"]).lower()
 d["metadata_event_start"]=pd.to_datetime(m["event_start"],errors="coerce")
 d["metadata_event_end"]=pd.to_datetime(m["event_end"],errors="coerce")
 d=d.sort_values(["time_stamp","source_row_order"],kind="stable").reset_index(drop=True)
 dt=d.time_stamp.diff()
 new_segment=dt.isna()|dt.gt(pd.Timedelta(minutes=10))|dt.le(pd.Timedelta(0))|d.train_test.ne(d.train_test.shift())
 d["segment_id"]=new_segment.cumsum().astype("int64")
 keys=list(zip(d.asset_id.astype(str),d.time_stamp))
 d["cross_event_split_overlap"]=[key in OVERLAP_KEYS for key in keys]
 miss=d[SENSORS].isna().sum(); zero=d[SENSORS].eq(0).sum()
 for col in SENSORS:
  sensor_counts[col]["missing"]+=int(miss[col]); sensor_counts[col]["zeros"]+=int(zero[col])
  train_values=d.loc[d.train_test.eq("train"),col].dropna()
  if len(train_values):
   sensor_counts[col]["train_count"]+=len(train_values)
   sensor_counts[col]["train_min"]=min(sensor_counts[col]["train_min"],float(train_values.min()))
   sensor_counts[col]["train_max"]=max(sensor_counts[col]["train_max"],float(train_values.max()))
 gaps=dt.gt(pd.Timedelta(minutes=10))
 out_path=EVENT_OUTPUT_DIR/f"{eid}.csv.gz"
 d.to_csv(out_path,index=False,compression="gzip")
 # Confirm the compressed output can be opened and has the expected row count.
 saved_rows=sum(1 for _ in pd.read_csv(out_path,compression="gzip",usecols=["id"],chunksize=100_000))
 saved_count=sum(len(x) for x in pd.read_csv(out_path,compression="gzip",usecols=["id"],chunksize=100_000))
 if saved_count!=len(d): raise ValueError(f"Saved row count mismatch for event {eid}")
 event_reports.append({"event_id":eid,"turbine":str(m["asset"]),"event_label":str(m["event_label"]).lower(),"rows":len(d),
  "train_rows":int(d.train_test.eq("train").sum()),"prediction_rows":int(d.train_test.eq("prediction").sum()),
  "missing_sensor_cells":int(miss.sum()),"non_numeric_sensor_cells":non_numeric_total,
  "duplicate_rows_in_raw":int(raw.duplicated().sum()),"duplicate_timestamps":int(d.time_stamp.duplicated().sum()),
  "gaps_over_10_minutes":int(gaps.sum()),"max_gap_minutes":float(dt[gaps].dt.total_seconds().div(60).max()) if gaps.any() else 0,
  "segments":int(d.segment_id.nunique()),"overlap_rows":int(d.cross_event_split_overlap.sum()),"output_file":str(out_path.relative_to(PROJECT_ROOT))})

EVENT_REPORT=pd.DataFrame(event_reports).sort_values("event_id",key=lambda s:s.astype(int))
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
EVENT_REPORT.to_csv(OUTPUT_DIR/"event_summary.csv",index=False)
sensor_report=[]
for col,stats in sensor_counts.items():
 lo,hi=stats["train_min"],stats["train_max"]
 sensor_report.append({"sensor_column":col,**stats,"constant_in_train":bool(stats["train_count"]>0 and lo==hi),"all_train_values_zero":bool(stats["train_count"]>0 and lo==0 and hi==0)})
SENSOR_REPORT=pd.DataFrame(sensor_report)
SENSOR_REPORT.to_csv(OUTPUT_DIR/"sensor_summary.csv",index=False)
print("Saved event files:",len(EVENT_REPORT))
print("Saved total rows:",int(EVENT_REPORT.rows.sum()))
print("Saved to:",OUTPUT_DIR)
print("Per-event summary:"); display(EVENT_REPORT)
print("Constant/all-zero sensors (reported, not removed):"); display(SENSOR_REPORT.loc[SENSOR_REPORT.constant_in_train|SENSOR_REPORT.all_train_values_zero])
print("Missing values remain blank; zeros remain unchanged; raw files were not modified.")

Saved event files: 22
Saved total rows: 1196747
Saved to: c:\Users\mohamad\Desktop\Predictive-Maintenance\data\processed\farm_a
Per-event summary:


,event_id,turbine,event_label,rows,train_rows,prediction_rows,missing_sensor_cells,non_numeric_sensor_cells,duplicate_rows_in_raw,duplicate_timestamps,gaps_over_10_minutes,max_gap_minutes,segments,overlap_rows,output_file
0,0,0,anomaly,54986,52148,2838,2,0,0,0,18,780.0,20,9730,data\processed\farm_a\events\0.csv.gz
1,3,10,normal,55487,52185,3302,4,0,0,0,41,780.0,43,8202,data\processed\farm_a\events\3.csv.gz
2,10,10,anomaly,53592,52323,1269,6,0,0,0,31,740.0,33,11054,data\processed\farm_a\events\10.csv.gz
3,13,21,normal,54010,50427,3583,0,0,0,0,34,13730.0,36,3583,data\processed\farm_a\events\13.csv.gz
4,14,13,normal,54197,52296,1901,0,0,0,0,15,780.0,17,1901,data\processed\farm_a\events\14.csv.gz
5,17,10,normal,55090,52309,2781,6,0,0,0,29,740.0,31,11054,data\processed\farm_a\events\17.csv.gz
6,22,21,anomaly,53036,51888,1148,2,0,0,0,24,1790.0,26,4731,data\processed\farm_a\events\22.csv.gz
7,24,0,normal,55003,52289,2714,0,0,0,0,11,780.0,13,5163,data\processed\farm_a\events\24.csv.gz
8,25,11,normal,54712,52289,2423,0,0,0,0,12,780.0,14,4197,data\processed\farm_a\events\25.csv.gz
9,26,0,anomaly,53702,52261,1441,2,0,0,0,19,740.0,21,9730,data\processed\farm_a\events\26.csv.gz


Constant/all-zero sensors (reported, not removed):


,sensor_column,missing,zeros,non_numeric,train_count,train_min,train_max,constant_in_train,all_train_values_zero
38,sensor_46,0,1196747,0,1146154,0.0,0.0,True,True
41,sensor_49,0,1196747,0,1146154,0.0,0.0,True,True


Missing values remain blank; zeros remain unchanged; raw files were not modified.


## 4. What to do next

The processed copy is in `data/processed/farm_a/`: one `.csv.gz` file per event in `events/`, plus `event_summary.csv` and `sensor_summary.csv`. Read a compressed event with `pd.read_csv(path, compression="gzip")`.

In `03_feature_engineering.ipynb`, build windows only within the same event, same `train_test` split, and same `segment_id`. Skip any candidate window with a missing sensor value. Fit any scaler using training rows only. Before reporting model metrics, use `cross_event_split_overlap` to prevent shared turbine/timestamps from leaking across training and prediction.

Known uncertainty remains: the README says 44 anomaly / 51 normal overall, while actual metadata says 45 / 50; Farm A metadata labels 12 anomaly / 10 normal. Some feature units are unclear. This preprocessing preserves those facts without guessing.